In [16]:
import pandas as pd

# Select a random sample of 10,000 rows from the Lending Club dataset
# This gives us different loan start dates instead of only the first 10,000 rows

In [17]:
import random
import pandas as pd

csv_path = "../data/raw/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

useful_columns = [
    "annual_inc",
    "loan_amnt",
    "installment",
    "issue_d",
    "out_prncp",
    "total_pymnt",
    "int_rate",
    "term",
    "hardship_flag"
]

# Total number of data rows in the dataset
total_rows = 2260701

# Randomly select 10,000 data-row positions
random.seed(42)
selected_rows = set(random.sample(range(1, total_rows + 1), 10000))

df_base = pd.read_csv(
    csv_path,
    usecols=useful_columns,
    skiprows=lambda x: x > 0 and x not in selected_rows
)

df_base = df_base.reset_index(drop=True)

df_base = df_base.rename(columns={
    "annual_inc": "annual_income",
    "loan_amnt": "loan_amount",
    "installment": "installment_amount",
    "issue_d": "start_date",
    "out_prncp": "outstanding_balance",
    "total_pymnt": "total_payment_amount",
    "int_rate": "interest_rate",
    "term": "loan_term",
    "hardship_flag": "hardship_label"
})

df_base.head()

,loan_amount,loan_term,interest_rate,installment_amount,annual_income,start_date,outstanding_balance,total_payment_amount,hardship_label
0,20000.0,36 months,9.17,637.58,95750.00,Dec-2015,0.0,7331.690000,N
1,19200.0,36 months,5.32,578.21,50000.00,Dec-2015,0.0,20801.128609,N
2,7000.0,36 months,5.32,210.81,16968.00,Dec-2015,0.0,7519.644958,N
3,9650.0,36 months,12.88,324.59,55500.00,Dec-2015,0.0,11677.101068,N
4,9600.0,36 months,6.49,294.19,64567.05,Dec-2015,0.0,10263.806046,N


In [18]:
# Check the different loan start dates in our random 10,000-row sample

df_base["start_date"].value_counts()

start_date
Mar-2016    275
Apr-2018    224
Aug-2017    210
Jul-2015    210
May-2018    200
           ... 
Aug-2007      1
May-2008      1
Aug-2008      1
Sep-2007      1
Jan-2010      1
Name: count, Length: 128, dtype: int64

In [19]:
df_base.shape

(10000, 9)

# Add synthetic customer information
# These values are dummy data for the project and are not real customer details.

In [26]:
df_base["full_name"] = "Customer " + (df_base.index + 1).astype(str)

df_base["email"] = (
    "customer" + (df_base.index + 1).astype(str) + "@example.com"
)

df_base["phone"] = (
    "+92-300-" + (df_base.index + 1).astype(str).str.zfill(7)
)

df_base[[
    "start_date",
    "full_name",
    "email",
    "phone"
]].sample(10, random_state=42)

,start_date,full_name,email,phone
6252,Nov-2018,Customer 6253,customer6253@example.com,+92-300-0006253
4684,Feb-2016,Customer 4685,customer4685@example.com,+92-300-0004685
1731,Jan-2015,Customer 1732,customer1732@example.com,+92-300-0001732
4742,Feb-2016,Customer 4743,customer4743@example.com,+92-300-0004743
4521,Mar-2016,Customer 4522,customer4522@example.com,+92-300-0004522
6340,Oct-2018,Customer 6341,customer6341@example.com,+92-300-0006341
576,Sep-2015,Customer 577,customer577@example.com,+92-300-0000577
5202,Oct-2014,Customer 5203,customer5203@example.com,+92-300-0005203
6363,Oct-2018,Customer 6364,customer6364@example.com,+92-300-0006364
439,Oct-2015,Customer 440,customer440@example.com,+92-300-0000440


# Create dummy payment status
# This represents whether the scheduled payment was received.

In [27]:
import numpy as np

np.random.seed(42)

df_base["payment_received"] = np.random.choice(
    ["Yes", "No"],
    size=len(df_base),
    p=[0.8, 0.2]
)

df_base["payment_received"].value_counts()

payment_received
Yes    8038
No     1962
Name: count, dtype: int64

# Create the current payment due date
# The first installment is due one month after the loan starts.
# The due date then advances monthly up to the reference date.

In [48]:
df_base["payment_due_date"] = (
    pd.to_datetime(df_base["start_date"], format="%b-%Y")
    + pd.DateOffset(months=1)
)

df_base[[
    "start_date",
    "payment_due_date"
]].sample(10, random_state=42)


,start_date,payment_due_date
6252,Nov-2018,2018-12-01
4684,Feb-2016,2016-03-01
1731,Jan-2015,2015-02-01
4742,Feb-2016,2016-03-01
4521,Mar-2016,2016-04-01
6340,Oct-2018,2018-11-01
576,Sep-2015,2015-10-01
5202,Oct-2014,2014-11-01
6363,Oct-2018,2018-11-01
439,Oct-2015,2015-11-01


# Create dummy payment received dates
# Paid customers are assigned a payment date on or after the due date.
# Unpaid customers have no payment received date.

In [35]:
df_base["payment_received_date"] = pd.NaT

paid_mask = df_base["payment_received"] == "Yes"

df_base.loc[paid_mask, "payment_received_date"] = (
    df_base.loc[paid_mask, "payment_due_date"]
    + pd.to_timedelta(
        np.random.randint(0, 6, size=paid_mask.sum()),
        unit="D"
    )
)

paid_sample = df_base[df_base["payment_received"] == "Yes"].sample(5, random_state=42)
unpaid_sample = df_base[df_base["payment_received"] == "No"].sample(5, random_state=42)

pd.concat([paid_sample, unpaid_sample])[[
    "payment_due_date",
    "payment_received",
    "payment_received_date"
]]

,payment_due_date,payment_received,payment_received_date
3546,2018-09-01,Yes,2018-09-06
925,2015-08-01,Yes,2015-08-03
4167,2017-06-01,Yes,2017-06-02
9773,2016-12-01,Yes,2016-12-05
1730,2015-02-01,Yes,2015-02-06
5516,2014-07-01,No,NaT
2843,2017-08-01,No,NaT
2211,2018-02-01,No,NaT
1870,2018-04-01,No,NaT
7455,2017-03-01,No,NaT


# Calculate days overdue
# For paid customers, calculate the days between the due date and payment date.
# For unpaid customers, calculate overdue days up to a fixed reference date.

In [64]:
# Calculate realistic days overdue for the current installment
# Paid customers are 0-5 days late.
# Unpaid customers are 1-30 days overdue.

np.random.seed(42)

df_base["days_overdue"] = 0

paid_mask = df_base["payment_received"] == "Yes"
unpaid_mask = df_base["payment_received"] == "No"

df_base.loc[paid_mask, "days_overdue"] = np.random.randint(
    0, 6, size=paid_mask.sum()
)

df_base.loc[unpaid_mask, "days_overdue"] = np.random.randint(
    1, 31, size=unpaid_mask.sum()
)

df_base[[
    "payment_received",
    "days_overdue"
]].sample(10, random_state=42)

,payment_received,days_overdue
6252,Yes,0
4684,Yes,3
1731,Yes,2
4742,Yes,2
4521,Yes,3
6340,Yes,1
576,Yes,3
5202,Yes,4
6363,Yes,1
439,Yes,0


In [37]:
# Check which start months are present in our 10,000-row sample
df_base["start_date"].value_counts().head(10)

start_date
Mar-2016    275
Apr-2018    224
Aug-2017    210
Jul-2015    210
May-2018    200
Nov-2017    191
Sep-2017    190
Jul-2017    190
Dec-2015    190
Oct-2018    187
Name: count, dtype: int64

# Calculate the current overdue amount
# An unpaid installment becomes overdue once its due date has passed.
# Paid installments have no remaining overdue amount.

In [65]:
df_base["overdue_amount"] = 0.0

overdue_mask = (
    (df_base["payment_received"] == "No") &
    (df_base["payment_due_date"] <= reference_date)
)

df_base.loc[overdue_mask, "overdue_amount"] = (
    df_base.loc[overdue_mask, "installment_amount"]
)

paid_sample = df_base[df_base["payment_received"] == "Yes"].sample(5, random_state=42)
unpaid_sample = df_base[df_base["payment_received"] == "No"].sample(5, random_state=42)

pd.concat([paid_sample, unpaid_sample])[[
    "installment_amount",
    "payment_due_date",
    "payment_received",
    "days_overdue",
    "overdue_amount"
]]

,installment_amount,payment_due_date,payment_received,days_overdue,overdue_amount
3546,242.77,2018-09-01,Yes,5,0.00
925,338.93,2015-08-01,Yes,1,0.00
4167,278.90,2017-06-01,Yes,0,0.00
9773,507.58,2016-12-01,Yes,1,0.00
1730,816.31,2015-02-01,Yes,3,0.00
5516,237.85,2014-07-01,No,13,237.85
2843,117.29,2017-08-01,No,27,117.29
2211,257.88,2018-02-01,No,10,257.88
1870,304.54,2018-04-01,No,27,304.54
7455,401.35,2017-03-01,No,29,401.35


# Get the number of scheduled monthly installments from the loan term

In [39]:
df_base["total_installments"] = (
    df_base["loan_term"].str.extract(r"(\d+)").astype(int)
)

df_base[[
    "loan_term",
    "total_installments"
]].sample(10, random_state=42)

,loan_term,total_installments
6252,36 months,36
4684,36 months,36
1731,60 months,60
4742,60 months,60
4521,36 months,36
6340,60 months,60
576,36 months,36
5202,36 months,36
6363,36 months,36
439,36 months,36


# Simulate late payments across the borrower's full loan term
# Each installment has an 20% chance of being late or unpaid.

In [52]:
np.random.seed(42)

max_installments = df_base["total_installments"].max()

late_history = np.random.choice(
    [0, 1],
    size=(len(df_base), max_installments),
    p=[0.96, 0.04]
)

df_base["total_late_payments"] = [
    late_history[i, :term].sum()
    for i, term in enumerate(df_base["total_installments"])
]

df_base[[
    "loan_term",
    "total_installments",
    "total_late_payments"
]].sample(10, random_state=42)

,loan_term,total_installments,total_late_payments
6252,36 months,36,2
4684,36 months,36,1
1731,60 months,60,3
4742,60 months,60,3
4521,36 months,36,1
6340,60 months,60,1
576,36 months,36,0
5202,36 months,36,3
6363,36 months,36,3
439,36 months,36,0


# Count how many borrowers have each number of late payments


In [53]:
df_base["total_late_payments"].value_counts().sort_index()

total_late_payments
0    1913
1    3077
2    2492
3    1501
4     663
5     246
6      64
7      33
8       5
9       6
Name: count, dtype: int64

# Create the continued delinquency label
# The label represents whether the borrower is likely to remain delinquent
# on the next installment.
# More late payments, a currently missed payment, and hardship increase the risk,
# but the label is not determined by any single feature.

In [68]:
# Create hardship label based on total late payments
# More than 3 late payments = hardship
# 3 or fewer late payments = no hardship

df_base["hardship_label"] = (
    df_base["total_late_payments"] > 3
).map({True: "Y", False: "N"})

# Show 5 Y and 5 N in a random mixed order

hardship_y = df_base[df_base["hardship_label"] == "Y"].sample(
    5, random_state=42
)

hardship_n = df_base[df_base["hardship_label"] == "N"].sample(
    5, random_state=42
)

hardship_sample = pd.concat([
    hardship_y,
    hardship_n
]).sample(frac=1, random_state=42)

hardship_sample[[
    "total_late_payments",
    "hardship_label"
]]

,total_late_payments,hardship_label
7122,2,N
297,4,Y
7870,1,N
9030,4,Y
840,3,N
4914,4,Y
8670,2,N
2169,4,Y
5022,4,Y
107,3,N


# Check the distribution of the continued delinquency label

In [59]:
df_base["hardship_label"].value_counts()

hardship_label
N    5000
Y    5000
Name: count, dtype: int64

# Show 5 random borrowers with all available columns


In [45]:
df_base.sample(5, random_state=42)

,loan_amount,loan_term,interest_rate,installment_amount,annual_income,start_date,outstanding_balance,total_payment_amount,hardship_label,full_name,email,phone,payment_received,payment_due_date,payment_received_date,days_overdue,overdue_amount,total_installments,total_late_payments,continued_delinquency_label
6252,25000.0,36 months,11.80,827.98,75000.00,Nov-2018,22636.84,3295.530000,N,Customer 6253,customer6253@example.com,+92-300-0006253,Yes,2018-12-01,2018-12-01,0,0.0,36,9,1
4684,23700.0,36 months,11.47,781.20,65000.00,Feb-2016,0.00,28138.017580,N,Customer 4685,customer4685@example.com,+92-300-0004685,Yes,2016-03-01,2016-03-03,2,0.0,36,7,1
1731,14525.0,60 months,11.99,323.03,48000.00,Jan-2015,0.00,17258.470000,N,Customer 1732,customer1732@example.com,+92-300-0001732,Yes,2015-02-01,2015-02-01,0,0.0,60,12,1
4742,35000.0,60 months,20.75,941.96,108436.35,Feb-2016,0.00,43766.359955,N,Customer 4743,customer4743@example.com,+92-300-0004743,Yes,2016-03-01,2016-03-02,1,0.0,60,8,1
4521,15000.0,36 months,5.32,451.73,100000.00,Mar-2016,0.00,16257.527562,N,Customer 4522,customer4522@example.com,+92-300-0004522,Yes,2016-04-01,2016-04-02,1,0.0,36,7,1


# Keep only the 16 required features for the final processed dataset


In [49]:
final_columns = [
    "full_name",
    "annual_income",
    "email",
    "phone",
    "loan_amount",
    "installment_amount",
    "start_date",
    "outstanding_balance",
    "overdue_amount",
    "days_overdue",
    "total_late_payments",
    "total_payment_amount",
    "interest_rate",
    "loan_term",
    "hardship_label",
    "continued_delinquency_label"
]

df_final = df_base[final_columns].copy()

# Save the 10,000-row processed dataset
df_final.to_csv(
    "../data/processed/hardship_data_10k.csv",
    index=False
)

print("Saved:", df_final.shape)

Saved: (10000, 16)


In [50]:
# Check the real hardship label distribution

df_base["hardship_label"].value_counts()

hardship_label
N    9998
Y       2
Name: count, dtype: int64

In [51]:
# Check the synthetic continued delinquency label distribution

df_base["continued_delinquency_label"].value_counts()

continued_delinquency_label
1    9969
0      31
Name: count, dtype: int64